In [18]:
# ==============================================================================
# AI-POWERED CONTRACT & LEGAL DOCUMENT RISK ANALYZER
# Teyzix Core Internship - Task AI-3
#
# HOW TO USE THIS FILE IN GOOGLE COLAB:
# This file is split into CELLS using "# %%" markers.
# Copy each section (between two "# %%" lines) into its own Colab cell,
# in the same order as they appear here, and run them one by one.
#
# Stack used: Python + Google Gemini API + Gradio (matches your setup)
# ==============================================================================

In [19]:
# %% cell 1
# Install all the packages we need.
# - google-generativeai  -> lets us talk to the Gemini API
# - gradio               -> builds our web UI
# - pypdf                -> reads text out of PDF files
# - python-docx          -> reads AND writes .docx (Word) files
# - reportlab            -> lets us generate PDF report files
# - numpy                -> used for math (cosine similarity in semantic search)
# -------------------------------------------------------------------------
!pip install -q google-generativeai gradio pypdf python-docx reportlab numpy


In [62]:
# %% CELL 2

# Imports + Gemini API key setup
# -------------------------------------------------------------------------
import os
import json
import uuid
import datetime
import numpy as np
import gradio as gr
import google.generativeai as genai

from pypdf import PdfReader
from docx import Document as DocxDocument
from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle
from reportlab.lib import colors
from google.colab import userdata

# ------------------------------------------------------------------------

 #--- PASTE YOUR GEMINI API KEY HERE ---
# Get a free key from https://aistudio.google.com/app/apikey
GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")

genai.configure(api_key=GEMINI_API_KEY)

# We use Gemini 2.5 Flash -> it's fast and cheap, good for this kind of task.
GEMINI_MODEL_NAME = "gemini-2.5-flash"
EMBEDDING_MODEL_NAME = "gemini-embedding-001"

text_model = genai.GenerativeModel(GEMINI_MODEL_NAME)


In [63]:
# %% CELL 3 -----------------------------------------------------------------
# IN-MEMORY "DATABASE"
#
# NOTE: The task asks for full user auth + a real database (PostgreSQL/SQLite)
# + an admin panel. That's a multi-service backend, which doesn't really fit
# a single Colab notebook (Colab resets every session, has no persistent DB
# server, and Gradio here runs as one shared demo, not a multi-user server).
#
# So instead, we simulate all of that with simple Python dictionaries/lists
# that live in memory while the notebook is running. This still lets us
# demonstrate: login, roles (user/admin), document history, and admin stats.
# If you wanted this in real production, you'd swap these lists for real
# SQLite/PostgreSQL tables - the rest of the AI logic stays the same.
# -------------------------------------------------------------------------

# Fake "users table" -> username : {password, role}
USERS_DB = {
    "admin": {"password": "admin123", "role": "admin"},
    "user": {"password": "user123", "role": "user"},
}

# This list holds every document we've analyzed in this session.
# Each item is a dictionary with everything about that document.
DOCUMENTS_DB = []

In [64]:
# %% CELL 4 -----------------------------------------------------------------
# STEP 1: DOCUMENT UPLOAD - reading text out of PDF / DOCX / TXT files
# -------------------------------------------------------------------------

def extract_text_from_file(file_path):
    """
    Takes the path of an uploaded file and returns its plain text content.
    Supports PDF, DOCX, and TXT. Raises a clear error for anything else.
    """
    extension = file_path.lower().split(".")[-1]

    if extension == "pdf":
        reader = PdfReader(file_path)
        text_parts = []
        for page in reader.pages:
            page_text = page.extract_text() or ""
            text_parts.append(page_text)
        return "\n".join(text_parts)

    elif extension == "docx":
        doc = DocxDocument(file_path)
        return "\n".join(paragraph.text for paragraph in doc.paragraphs)

    elif extension == "txt":
        with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
            return f.read()

    else:
        raise ValueError(f"Unsupported file type: .{extension}. Please upload PDF, DOCX, or TXT.")


def validate_document_text(text):
    """
    Basic sanity check before we send anything to the AI.
    Makes sure the document isn't empty or too short to be a real contract.
    """
    if text is None or len(text.strip()) < 50:
        raise ValueError("This document seems empty or too short to analyze. Please check the file.")
    return True

In [65]:
# %% CELL 5 -----------------------------------------------------------------
# HELPER: a safe wrapper around calling Gemini and getting clean JSON back.
#
# Gemini sometimes wraps JSON in ```json ... ``` markdown fences, so we
# strip those before parsing. We also catch errors so one bad response
# doesn't crash the whole app.
# -------------------------------------------------------------------------

def ask_gemini_for_json(prompt):
    """
    Sends a prompt to Gemini and tries to parse the reply as JSON.
    Returns a Python dict/list, or an error dict if something goes wrong.
    """
    try:
        response = text_model.generate_content(prompt)
        raw_text = response.text.strip()

        # Remove markdown code fences like ```json ... ``` if present
        if raw_text.startswith("```"):
            raw_text = raw_text.strip("`")
            raw_text = raw_text.replace("json", "", 1).strip()

        return json.loads(raw_text)

    except json.JSONDecodeError:
        # The model didn't return valid JSON - return the raw text so we can debug
        return {"error": "Could not parse AI response as JSON", "raw_response": raw_text}
    except Exception as e:
        return {"error": str(e)}


def ask_gemini_for_text(prompt):
    """Simple wrapper for when we just want plain text back (no JSON)."""
    try:
        response = text_model.generate_content(prompt)
        return response.text.strip()
    except Exception as e:
        return f"Error generating response: {e}"

In [66]:
# %% CELL 6-----------------------------------------------------------------
# STEP 2: AI DOCUMENT ANALYSIS
# Extracts contract type, parties, dates, clauses etc. as structured JSON.
# -------------------------------------------------------------------------

def analyze_document(document_text):
   """
    Asks Gemini to read the contract and pull out the key structured info.
    Returns a dictionary matching the fields the task asks for.
    """
   prompt = f"""
You are a legal document analysis AI. Read the contract text below and extract
the following information. If something is not mentioned in the document,
use the value "Not specified".

Return ONLY valid JSON (no markdown, no explanation) in exactly this format:

{{
  "contract_type": "string",
  "parties_involved": ["list of party names"],
  "effective_date": "string",
  "expiry_date": "string",
  "payment_terms": "short description",
  "renewal_clause": "short description",
  "confidentiality_clause": "short description",
  "termination_clause": "short description",
  "responsibilities": ["list of key responsibilities for each party"]
}}

CONTRACT TEXT:
{document_text[:15000]}
"""
   return ask_gemini_for_json(prompt)

In [67]:
# %% CELL 7 -----------------------------------------------------------------
# STEP 3: AI RISK DETECTION
# Finds missing clauses, red flags, ambiguous wording, unusual payment terms.
# Every risk comes with a confidence score (0-100) and a plain explanation.
# -------------------------------------------------------------------------

def detect_risks(document_text):
    """
    Asks Gemini to act like a cautious lawyer and flag risky/missing things.
    Returns a list of risk dictionaries.
    """
    prompt = f"""
You are an AI legal risk analyst. Carefully review the contract text below and
identify potential risks. Look specifically for:
- Missing clauses that a normal contract of this type should have
- High-risk conditions (e.g. one-sided obligations, unlimited liability)
- Ambiguous or vague statements
- Unusual payment terms
- Any other legal red flags

Return ONLY valid JSON: a list of risk objects, each in exactly this format:

[
  {{
    "risk_type": "Missing Clause | High-Risk Condition | Ambiguous Statement | Unusual Payment Terms | Legal Red Flag",
    "title": "short title of the issue",
    "explanation": "plain-English explanation of why this is risky",
    "severity": "Low | Medium | High",
    "confidence_score": 0
  }}
]

confidence_score must be an integer from 0 to 100, representing how sure you
are that this is a real issue. If the contract looks completely fine with no
issues, return an empty list [].

CONTRACT TEXT:
\"\"\"{document_text[:15000]}\"\"\"
"""
    result = ask_gemini_for_json(prompt)
    # Make sure we always return a list, even if something went wrong
    if isinstance(result, list):
        return result
    return []


def calculate_overall_risk_score(risks):
    """
    Turns the list of individual risks into one overall score (0-100)
    for the document, weighted by severity.
    High severity issues count more toward the final score.
    """
    if not risks:
        return 0

    severity_weight = {"Low": 1, "Medium": 2, "High": 3}
    total_weighted = 0
    total_weight = 0

    for risk in risks:
        weight = severity_weight.get(risk.get("severity", "Low"), 1)
        confidence = risk.get("confidence_score", 0)
        total_weighted += weight * confidence
        total_weight += weight

    if total_weight == 0:
        return 0

    return round(total_weighted / total_weight, 1)

In [68]:
# %% CELL 8 -----------------------------------------------------------------
# STEP 4: AI SUMMARY
# Executive summary, key obligations, important dates/clauses, recommended actions.
# -------------------------------------------------------------------------

def generate_summary(document_text):
    """
    Asks Gemini to summarize the contract in a business-friendly way.
    """
    prompt = f"""
You are an AI assistant helping a busy business owner understand a contract
without reading the whole thing. Read the contract below and produce a summary.

Return ONLY valid JSON in exactly this format:

{{
  "executive_summary": "2-4 sentence plain-English overview of the contract",
  "key_obligations": ["list of the main things each party must do"],
  "important_dates": ["list of important dates and what they mean"],
  "important_clauses": ["list of the most important clauses to be aware of"],
  "recommended_actions": ["list of practical next steps or things to negotiate/review"]
}}

CONTRACT TEXT:
\"\"\"{document_text[:15000]}\"\"\"
"""
    return ask_gemini_for_json(prompt)


In [69]:
# %% CELL 9 -----------------------------------------------------------------
# STEP 5: SEMANTIC SEARCH (mini RAG pipeline)
#
# How this works, in plain terms:
# 1. We cut the document into small overlapping "chunks" of text.
# 2. We turn each chunk into a list of numbers (an "embedding") using Gemini.
#    Similar meaning = similar numbers.
# 3. When the user asks a question, we embed the question too, and find the
#    chunks whose numbers are closest to the question's numbers (cosine similarity).
# 4. We hand only those relevant chunks + the question to Gemini and ask it
#    to answer using just that context. This is called "RAG" (Retrieval
#    Augmented Generation) - it keeps answers grounded in the real document.
# -------------------------------------------------------------------------

def chunk_text(text, chunk_size=800, overlap=100):
    """
    Splits a long document into smaller overlapping pieces.
    Overlap helps make sure we don't accidentally cut a sentence/clause
    in half right at a chunk boundary.
    """
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start = end - overlap  # step back a bit so chunks overlap
    return chunks


def embed_text(text):
    """Turns a piece of text into an embedding vector (list of numbers)."""
    result = genai.embed_content(model=EMBEDDING_MODEL_NAME, content=text)
    return np.array(result["embedding"])


def cosine_similarity(vec_a, vec_b):
    """Standard similarity measure between two vectors: 1 = identical meaning, 0 = unrelated."""
    return np.dot(vec_a, vec_b) / (np.linalg.norm(vec_a) * np.linalg.norm(vec_b) + 1e-10)


def build_search_index(document_text):
    """
    Prepares a document for semantic search by chunking it and embedding
    every chunk. Returns (chunks, embeddings) to be stored with the document.
    """
    chunks = chunk_text(document_text)
    embeddings = [embed_text(chunk) for chunk in chunks]
    return chunks, embeddings


def semantic_search(query, chunks, embeddings, top_k=3):
    """
    Finds the most relevant chunks for a natural-language query,
    then asks Gemini to answer the query using only those chunks.
    """
    query_vector = embed_text(query)
    scores = [cosine_similarity(query_vector, chunk_vec) for chunk_vec in embeddings]

    # Get indices of the top_k most similar chunks
    top_indices = np.argsort(scores)[::-1][:top_k]
    relevant_chunks = [chunks[i] for i in top_indices]
    context = "\n---\n".join(relevant_chunks)

    prompt = f"""
Answer the user's question using ONLY the context below, which is taken from
a legal contract. If the answer isn't in the context, say so honestly.

CONTEXT:
\"\"\"{context}\"\"\"

QUESTION: {query}

Give a clear, direct answer in plain English.
"""
    return ask_gemini_for_text(prompt)



In [70]:
# %% CELL 10---------------------------------------------------------------
# STEP 6 & 7: REPORT GENERATION (PDF + DOCX)
# -------------------------------------------------------------------------

def generate_pdf_report(doc_record, output_path):
    """
    Builds a nicely formatted PDF report for one analyzed document,
    using the reportlab library.
    """
    pdf = SimpleDocTemplate(output_path, pagesize=A4)
    styles = getSampleStyleSheet()
    story = []

    story.append(Paragraph(f"Contract Risk Analysis Report", styles["Title"]))
    story.append(Paragraph(f"Document: {doc_record['filename']}", styles["Normal"]))
    story.append(Paragraph(f"Analyzed on: {doc_record['upload_date']}", styles["Normal"]))
    story.append(Spacer(1, 16))

    # --- Executive Summary ---
    story.append(Paragraph("Executive Summary", styles["Heading2"]))
    story.append(Paragraph(doc_record["summary"].get("executive_summary", "N/A"), styles["Normal"]))
    story.append(Spacer(1, 12))

    # --- Contract Details ---
    story.append(Paragraph("Contract Details", styles["Heading2"]))
    analysis = doc_record["analysis"]
    details_table_data = [
        ["Contract Type", analysis.get("contract_type", "N/A")],
        ["Parties Involved", ", ".join(analysis.get("parties_involved", []))],
        ["Effective Date", analysis.get("effective_date", "N/A")],
        ["Expiry Date", analysis.get("expiry_date", "N/A")],
        ["Payment Terms", analysis.get("payment_terms", "N/A")],
        ["Renewal Clause", analysis.get("renewal_clause", "N/A")],
        ["Confidentiality Clause", analysis.get("confidentiality_clause", "N/A")],
        ["Termination Clause", analysis.get("termination_clause", "N/A")],
    ]
    details_table = Table(details_table_data, colWidths=[150, 330])
    details_table.setStyle(TableStyle([
        ("GRID", (0, 0), (-1, -1), 0.5, colors.grey),
        ("BACKGROUND", (0, 0), (0, -1), colors.whitesmoke),
        ("VALIGN", (0, 0), (-1, -1), "TOP"),
        ("FONTSIZE", (0, 0), (-1, -1), 9),
    ]))
    story.append(details_table)
    story.append(Spacer(1, 12))

    # --- Risk Assessment ---
    story.append(Paragraph(f"AI Risk Assessment (Overall Score: {doc_record['risk_score']}/100)", styles["Heading2"]))
    if doc_record["risks"]:
        risk_table_data = [["Type", "Title", "Severity", "Confidence"]]
        for risk in doc_record["risks"]:
            risk_table_data.append([
                risk.get("risk_type", ""),
                risk.get("title", ""),
                risk.get("severity", ""),
                f"{risk.get('confidence_score', 0)}%",
            ])
        risk_table = Table(risk_table_data, colWidths=[100, 180, 70, 70])
        risk_table.setStyle(TableStyle([
            ("GRID", (0, 0), (-1, -1), 0.5, colors.grey),
            ("BACKGROUND", (0, 0), (-1, 0), colors.HexColor("#4472C4")),
            ("TEXTCOLOR", (0, 0), (-1, 0), colors.white),
            ("FONTSIZE", (0, 0), (-1, -1), 8),
        ]))
        story.append(risk_table)
        story.append(Spacer(1, 8))
        for risk in doc_record["risks"]:
            story.append(Paragraph(f"<b>{risk.get('title')}</b>: {risk.get('explanation')}", styles["Normal"]))
    else:
        story.append(Paragraph("No significant risks detected.", styles["Normal"]))
    story.append(Spacer(1, 12))

    # --- Recommendations ---
    story.append(Paragraph("Recommended Actions", styles["Heading2"]))
    for action in doc_record["summary"].get("recommended_actions", []):
        story.append(Paragraph(f"- {action}", styles["Normal"]))

    pdf.build(story)
    return output_path


def generate_docx_report(doc_record, output_path):
    """Builds the same report as above, but as a Word (.docx) file."""
    doc = DocxDocument()
    doc.add_heading("Contract Risk Analysis Report", 0)
    doc.add_paragraph(f"Document: {doc_record['filename']}")
    doc.add_paragraph(f"Analyzed on: {doc_record['upload_date']}")

    doc.add_heading("Executive Summary", level=1)
    doc.add_paragraph(doc_record["summary"].get("executive_summary", "N/A"))

    doc.add_heading("Contract Details", level=1)
    analysis = doc_record["analysis"]
    for label, key in [
        ("Contract Type", "contract_type"),
        ("Effective Date", "effective_date"),
        ("Expiry Date", "expiry_date"),
        ("Payment Terms", "payment_terms"),
        ("Renewal Clause", "renewal_clause"),
        ("Confidentiality Clause", "confidentiality_clause"),
        ("Termination Clause", "termination_clause"),
    ]:
        doc.add_paragraph(f"{label}: {analysis.get(key, 'N/A')}")
    doc.add_paragraph("Parties Involved: " + ", ".join(analysis.get("parties_involved", [])))

    doc.add_heading(f"AI Risk Assessment (Overall Score: {doc_record['risk_score']}/100)", level=1)
    for risk in doc_record["risks"]:
        doc.add_paragraph(
            f"[{risk.get('severity')}] {risk.get('title')} "
            f"(Confidence: {risk.get('confidence_score')}%) - {risk.get('explanation')}"
        )

    doc.add_heading("Recommended Actions", level=1)
    for action in doc_record["summary"].get("recommended_actions", []):
        doc.add_paragraph(f"- {action}", style="List Bullet")

    doc.save(output_path)
    return output_path

In [71]:
# %% CELL 11 ----------------------------------------------------------------
# THE MAIN "ANALYZE DOCUMENT" PIPELINE
# This ties every AI step together into one function the UI can call.
# -------------------------------------------------------------------------

def run_full_analysis_pipeline(file_path, filename, uploaded_by):
    """
    Runs the complete pipeline on one uploaded document:
    extract text -> validate -> analyze -> detect risks -> summarize ->
    build search index -> save everything into DOCUMENTS_DB (our "database")
    """
    text = extract_text_from_file(file_path)
    validate_document_text(text)

    analysis = analyze_document(text)
    risks = detect_risks(text)
    risk_score = calculate_overall_risk_score(risks)
    summary = generate_summary(text)
    chunks, embeddings = build_search_index(text)

    doc_record = {
        "id": str(uuid.uuid4())[:8],
        "filename": filename,
        "uploaded_by": uploaded_by,
        "upload_date": datetime.datetime.now().strftime("%Y-%m-%d %H:%M"),
        "raw_text": text,
        "analysis": analysis,
        "risks": risks,
        "risk_score": risk_score,
        "summary": summary,
        "chunks": chunks,
        "embeddings": embeddings,
    }

    DOCUMENTS_DB.append(doc_record)
    return doc_record

In [72]:
# %% CELL 12 ----------------------------------------------------------------
# FORMATTING HELPERS - turn our data into nice Markdown for display in Gradio
# -------------------------------------------------------------------------

def format_analysis_markdown(analysis):
    if "error" in analysis:
        return f"⚠️ Error analyzing document: {analysis['error']}"
    return f"""
### 📄 Contract Details
| Field | Value |
|---|---|
| **Contract Type** | {analysis.get('contract_type', 'N/A')} |
| **Parties Involved** | {', '.join(analysis.get('parties_involved', []))} |
| **Effective Date** | {analysis.get('effective_date', 'N/A')} |
| **Expiry Date** | {analysis.get('expiry_date', 'N/A')} |
| **Payment Terms** | {analysis.get('payment_terms', 'N/A')} |
| **Renewal Clause** | {analysis.get('renewal_clause', 'N/A')} |
| **Confidentiality Clause** | {analysis.get('confidentiality_clause', 'N/A')} |
| **Termination Clause** | {analysis.get('termination_clause', 'N/A')} |

**Responsibilities:**
{chr(10).join('- ' + r for r in analysis.get('responsibilities', []))}
"""
def format_risks_markdown(risks, risk_score):
    if not risks:
        return f"### 🟢 Overall Risk Score: {risk_score}/100\nNo significant risks detected."

    severity_emoji = {"High": "🔴", "Medium": "🟠", "Low": "🟡"}
    lines = [f"### ⚠️ Overall Risk Score: {risk_score}/100\n"]
    for risk in risks:
        emoji = severity_emoji.get(risk.get("severity", "Low"), "⚪")
        lines.append(
            f"{emoji} **{risk.get('title')}** ({risk.get('risk_type')}) "
            f"- Confidence: {risk.get('confidence_score')}%\n\n"
            f"> {risk.get('explanation')}\n"
        )
    return "\n".join(lines)


def format_summary_markdown(summary):
    if "error" in summary:
        return f"⚠️ Error generating summary: {summary['error']}"
    return f"""
### 📝 Executive Summary
{summary.get('executive_summary', 'N/A')}

**Key Obligations:**
{chr(10).join('- ' + o for o in summary.get('key_obligations', []))}

**Important Dates:**
{chr(10).join('- ' + d for d in summary.get('important_dates', []))}

**Important Clauses:**
{chr(10).join('- ' + c for c in summary.get('important_clauses', []))}

**Recommended Actions:**
{chr(10).join('- ' + a for a in summary.get('recommended_actions', []))}
"""


def build_dashboard_markdown():
    """Builds the AI Insights Dashboard text shown in the Dashboard tab."""
    if not DOCUMENTS_DB:
        return "No documents analyzed yet. Upload one in the 'Analyze Document' tab first."

    total_docs = len(DOCUMENTS_DB)
    avg_risk = round(sum(d["risk_score"] for d in DOCUMENTS_DB) / total_docs, 1)
    high_risk_docs = sum(1 for d in DOCUMENTS_DB if d["risk_score"] >= 60)

    # Count how often each risk_type shows up across all documents
    risk_counter = {}
    for d in DOCUMENTS_DB:
        for r in d["risks"]:
            risk_type = r.get("risk_type", "Unknown")
            risk_counter[risk_type] = risk_counter.get(risk_type, 0) + 1
    top_risks = sorted(risk_counter.items(), key=lambda x: x[1], reverse=True)[:5]
    top_risks_text = "\n".join(f"- {name}: {count} occurrence(s)" for name, count in top_risks) or "None yet"

    return f"""
### 📊 AI Insights Dashboard
- **Total Documents Analyzed:** {total_docs}
- **Average Risk Score:** {avg_risk}/100
- **High-Risk Documents (score ≥ 60):** {high_risk_docs}

**Frequently Detected Risk Types:**
{top_risks_text}
"""


def build_history_table():
    """Builds a table (list of lists) for the Document History tab."""
    return [
        [d["id"], d["filename"], d["uploaded_by"], d["upload_date"], d["risk_score"]]
        for d in DOCUMENTS_DB
    ]

In [73]:
# %% CELL 13 ----------------------------------------------------------------
# GRADIO UI - all the tabs/pages the user interacts with
# -------------------------------------------------------------------------

def do_login(username, password):
    """Checks username/password against our simple USERS_DB."""
    user = USERS_DB.get(username)
    if user and user["password"] == password:
        return f"✅ Logged in as **{username}** (role: {user['role']})", username, user["role"]
    return "❌ Invalid username or password.", None, None


def do_analyze(file_obj, username):
    """Called when the user clicks 'Analyze Document'."""
    if not username:
        return "⚠️ Please log in first (Login tab).", "", "", None
    if file_obj is None:
        return "⚠️ Please upload a document first.", "", "", None

    try:
        doc_record = run_full_analysis_pipeline(file_obj.name, os.path.basename(file_obj.name), username)
    except Exception as e:
        return f"⚠️ Error: {e}", "", "", None

    analysis_md = format_analysis_markdown(doc_record["analysis"])
    risk_md = format_risks_markdown(doc_record["risks"], doc_record["risk_score"])
    summary_md = format_summary_markdown(doc_record["summary"])

    # Return the doc id too, so other tabs (search/reports) know which doc to use
    return analysis_md, risk_md, summary_md, doc_record["id"]


def get_doc_choices():
    """Returns dropdown-friendly labels for every analyzed document."""
    return [f"{d['id']} - {d['filename']}" for d in DOCUMENTS_DB]


def find_doc_by_choice(choice):
    """Looks up the full document record from a dropdown label like 'ab12cd - file.pdf'."""
    if not choice:
        return None
    doc_id = choice.split(" - ")[0]
    return next((d for d in DOCUMENTS_DB if d["id"] == doc_id), None)


def do_semantic_search(doc_choice, query):
    doc = find_doc_by_choice(doc_choice)
    if doc is None:
        return "⚠️ Please select a document first."
    if not query.strip():
        return "⚠️ Please type a question."
    return semantic_search(query, doc["chunks"], doc["embeddings"])


def do_generate_report(doc_choice, report_format):
    doc = find_doc_by_choice(doc_choice)
    if doc is None:
        return None
    safe_name = doc["filename"].rsplit(".", 1)[0]
    if report_format == "PDF":
        output_path = f"/content/{safe_name}_report.pdf"
        generate_pdf_report(doc, output_path)
    else:
        output_path = f"/content/{safe_name}_report.docx"
        generate_docx_report(doc, output_path)
    return output_path


def refresh_dashboard():
    return build_dashboard_markdown()


def refresh_history():
    return build_history_table()


def refresh_dropdowns():
    choices = get_doc_choices()
    return gr.update(choices=choices), gr.update(choices=choices)


def admin_view_stats(role):
    """Only shows real stats if the logged-in user's role is 'admin'."""
    if role != "admin":
        return "⛔ Access denied. Admin role required."
    lines = ["### 🛠️ Admin Panel - System Overview\n"]
    lines.append(f"**Total registered users:** {len(USERS_DB)}")
    lines.append(f"**Total documents processed:** {len(DOCUMENTS_DB)}\n")
    lines.append("**Registered Users:**")
    for uname, info in USERS_DB.items():
        lines.append(f"- {uname} ({info['role']})")
    lines.append("\n**Processing Log:**")
    for d in DOCUMENTS_DB:
        lines.append(f"- [{d['upload_date']}] {d['filename']} uploaded by {d['uploaded_by']} | Risk: {d['risk_score']}")
    return "\n".join(lines)


# ----- Build the actual interface -----
with gr.Blocks(title="AI Contract Risk Analyzer") as app:
    gr.Markdown("# ⚖️ AI-Powered Contract & Legal Document Risk Analyzer")
    gr.Markdown("Teyzix Core Internship - Task AI-3")

    # We store the logged-in username/role in Gradio "State" so every tab can see it
    logged_in_user = gr.State(None)
    logged_in_role = gr.State(None)

    with gr.Tab("🔐 Login"):
        gr.Markdown("Demo accounts: **admin / admin123** (admin role) or **user / user123** (normal user)")
        username_box = gr.Textbox(label="Username")
        password_box = gr.Textbox(label="Password", type="password")
        login_btn = gr.Button("Login", variant="primary")
        login_status = gr.Markdown()
        login_btn.click(
            do_login,
            inputs=[username_box, password_box],
            outputs=[login_status, logged_in_user, logged_in_role],
        )

    with gr.Tab("📤 Analyze Document"):
        gr.Markdown("Upload a contract (PDF, DOCX, or TXT) to analyze it with AI.")
        file_input = gr.File(label="Upload Document", file_types=[".pdf", ".docx", ".txt"])
        analyze_btn = gr.Button("🔍 Analyze with AI", variant="primary")
        current_doc_id = gr.State(None)  # remembers which doc was just analyzed

        analysis_output = gr.Markdown(label="Contract Analysis")
        risk_output = gr.Markdown(label="Risk Assessment")
        summary_output = gr.Markdown(label="AI Summary")

        analyze_btn.click(
            do_analyze,
            inputs=[file_input, logged_in_user],
            outputs=[analysis_output, risk_output, summary_output, current_doc_id],
        )

    with gr.Tab("💬 Semantic Search"):
        gr.Markdown("Ask natural-language questions about any analyzed document.")
        search_doc_dropdown = gr.Dropdown(label="Select Document", choices=[])
        search_query_box = gr.Textbox(label="Your Question", placeholder="e.g. What are the payment terms?")
        search_btn = gr.Button("Search", variant="primary")
        search_result = gr.Markdown()
        search_btn.click(do_semantic_search, inputs=[search_doc_dropdown, search_query_box], outputs=search_result)

    with gr.Tab("📊 Dashboard"):
        dashboard_output = gr.Markdown()
        dashboard_refresh_btn = gr.Button("🔄 Refresh Dashboard")
        dashboard_refresh_btn.click(refresh_dashboard, outputs=dashboard_output)

    with gr.Tab("🗂️ Document History"):
        history_table = gr.Dataframe(
            headers=["Doc ID", "Filename", "Uploaded By", "Date", "Risk Score"],
            label="Previously Analyzed Documents",
        )
        history_refresh_btn = gr.Button("🔄 Refresh History")
        history_refresh_btn.click(refresh_history, outputs=history_table)

    with gr.Tab("📑 Reports"):
        report_doc_dropdown = gr.Dropdown(label="Select Document", choices=[])
        report_format_radio = gr.Radio(["PDF", "DOCX"], value="PDF", label="Report Format")
        report_btn = gr.Button("Generate Report", variant="primary")
        report_file_output = gr.File(label="Download Report")
        report_btn.click(
            do_generate_report,
            inputs=[report_doc_dropdown, report_format_radio],
            outputs=report_file_output,
        )

    with gr.Tab("🛠️ Admin Panel"):
        gr.Markdown("Only accessible to users with the admin role.")
        admin_refresh_btn = gr.Button("🔄 View System Stats")
        admin_output = gr.Markdown()
        admin_refresh_btn.click(admin_view_stats, inputs=[logged_in_role], outputs=admin_output)

    # Whenever a new document is analyzed, refresh the dropdowns in Search & Reports tabs
    current_doc_id.change(refresh_dropdowns, outputs=[search_doc_dropdown, report_doc_dropdown])


In [ ]:
# %% CELL 14 ----------------------------------------------------------------
# LAUNCH THE APP
# -------------------------------------------------------------------------
app.launch(share=True, debug=True)


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://1508d1c071dc25092e.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


/usr/local/lib/python3.12/dist-packages/gradio/routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
/usr/local/lib/python3.12/dist-packages/gradio/routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
/usr/local/lib/python3.12/dist-packages/gradio/routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
/usr/local/lib/python3.12/dist-packages/gradio/routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
/usr/local/lib/python3.12/di